In [ ]:
import requests
import os
import re
from openai import OpenAI
from tavily import TavilyClient
from dotenv import load_dotenv

# Загружаем переменные окружения
load_dotenv()

# Настройка API-ключей
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_ID = os.getenv("MODEL_ID")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

# Системный промпт
AGENT_SYSTEM_PROMPT = """
Ты умный туристический помощник. Твоя задача — анализировать запросы пользователя и решать задачи шаг за шагом с помощью доступных инструментов.

# Доступные инструменты:
- `get_weather(city: str)`: Запрос реальной погоды для указанного города.
- `get_attraction(city: str, weather: str)`: Поиск рекомендуемых достопримечательностей по городу и погоде.

# Требования к формату вывода:
Каждый твой ответ должен строго следовать этому формату — одна пара Thought и Action:

Thought: [Твой процесс мышления и план следующего шага]
Action: [Конкретное действие, которое ты хочешь выполнить]

Формат Action должен быть одним из следующих:
1. Вызов инструмента: function_name(arg_name=\"arg_value\")
2. Завершение задачи: Finish[окончательный ответ]

# Важные замечания:
- Выводи только одну пару Thought-Action за раз
- Action должен быть на одной строке, без переносов строки
- Когда соберёшь достаточно информации для ответа, обязательно используй формат Action: Finish[окончательный ответ]

Начнём!
"""


In [ ]:
def get_weather(city: str) -> str:
    """
    Запрашивает реальную информацию о погоде через API wttr.in.
    """
    # Точка API — запрашиваем данные в формате JSON
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        # Выполняем сетевой запрос
        response = requests.get(url)
        # Проверяем код ответа (200 = успех)
        response.raise_for_status()
        # Разбираем возвращённые JSON-данные
        data = response.json()
        
        # Извлекаем текущие погодные условия
        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']
        
        # Форматируем в естественный язык и возвращаем
        return f"{city} текущая погода: {weather_desc}, температура {temp_c} °C"
        
    except requests.exceptions.RequestException as e:
        # Обработка сетевых ошибок
        return f"Ошибка: проблема с сетью при запросе погоды — {e}"
    except (KeyError, IndexError) as e:
        # Обработка ошибок разбора данных
        return f"Ошибка: не удалось разобрать данные о погоде, возможно неверное название города — {e}"

def get_attraction(city: str, weather: str) -> str:
    """
    Ищет и возвращает рекомендации достопримечательностей с помощью Tavily Search API
    на основе города и погоды.
    """
    api_key = os.environ.get("TAVILY_API_KEY")

    if not api_key:
        return "Ошибка: переменная окружения TAVILY_API_KEY не настроена."

    # Инициализируем клиент Tavily
    tavily = TavilyClient(api_key=api_key)
    
    # Формируем точный поисковый запрос
    query = f"Самые стоящие достопримечательности '{city}' в погоду '{weather}' и причины посетить"
    
    try:
        # Вызываем API; include_answer=True возвращает сводный ответ
        response = tavily.search(query=query, search_depth="basic", include_answer=True)
        
        # Результаты Tavily уже достаточно чистые — используем напрямую
        if response.get("answer"):
            return response["answer"]
        
        # Если сводного ответа нет — форматируем сырые результаты
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        if not formatted_results:
             return "К сожалению, рекомендации по достопримечательностям не найдены."

        return "По результатам поиска найдено следующее:\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"Ошибка: проблема при выполнении поиска через Tavily — {e}"

# Все инструменты в одном словаре для удобного вызова
available_tools = {
    "get_weather": get_weather,
    "get_attraction": get_attraction,
}
print("✅ Инструменты определены!")


In [ ]:
class OpenAICompatibleClient:
    """
    Клиент для вызова любого LLM-сервиса, совместимого с интерфейсом OpenAI.
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        """Вызывает LLM API для генерации ответа."""
        print("Вызов языковой модели...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("Языковая модель успешно ответила.")
            return answer
        except Exception as e:
            print(f"Ошибка при вызове LLM API: {e}")
            return "Ошибка: проблема при обращении к языковой модели."

class TravelAssistant:
    """
    Класс умного туристического помощника.
    """
    def __init__(self):
        self.llm = OpenAICompatibleClient(
            model=MODEL_ID,
            api_key=API_KEY,
            base_url=BASE_URL
        )
        self.prompt_history = []
    
    def reset(self):
        """Сбрасывает историю диалога."""
        self.prompt_history = []
    
    def add_user_message(self, message: str):
        """Добавляет сообщение пользователя в историю."""
        self.prompt_history.append(f"Запрос пользователя: {message}")
    
    def add_assistant_message(self, message: str):
        """Добавляет сообщение ассистента в историю."""
        self.prompt_history.append(message)
    
    def add_observation(self, observation: str):
        """Добавляет результат наблюдения в историю."""
        self.prompt_history.append(f"Observation: {observation}")
print("✅ Класс ассистента определён!")


In [ ]:
def display_conversation(history):
    """Красиво отображает историю диалога."""
    print("\n" + "="*60)
    print("📝 История диалога")
    print("="*60)
    
    for i, message in enumerate(history, 1):
        if message.startswith("Запрос пользователя:"):
            print(f"\n👤 Пользователь [{i}]: {message[21:]}")
        elif message.startswith("Thought:"):
            print(f"\n🤔 Мысль [{i}]: {message[8:].strip()}")
        elif message.startswith("Action:"):
            print(f"🛠️  Действие [{i}]: {message[7:].strip()}")
        elif message.startswith("Observation:"):
            print(f"📊 Наблюдение [{i}]: {message[12:].strip()}")
        else:
            print(f"💬 Сообщение [{i}]: {message}")
    
    print("="*60 + "\n")


In [ ]:
def run_assistant(user_input, max_iterations=5, display=True):
    """
    Основная функция запуска туристического помощника.

    Args:
        user_input: вопрос пользователя
        max_iterations: максимальное число итераций цикла
        display: отображать ли историю диалога

    Returns:
        tuple: (финальный ответ, полная история диалога)
    """
    assistant = TravelAssistant()
    assistant.add_user_message(user_input)

    if display:
        print(f"👤 Ввод пользователя: {user_input}")
        print("="*50)

    for i in range(max_iterations):
        if display:
            print(f"\n🔄 Итерация {i+1}/{max_iterations}")

        # Формируем полный промпт и вызываем LLM
        full_prompt = "\n".join(assistant.prompt_history)
        llm_output = assistant.llm.generate(full_prompt, AGENT_SYSTEM_PROMPT)
        # Обрезаем лишние пары Thought-Action
        match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
        if match:
            truncated = match.group(1).strip()
            if truncated != llm_output.strip():
                llm_output = truncated
                print("⚠️ Лишние пары Thought-Action обрезаны")

        assistant.add_assistant_message(llm_output)

        if display:
            print(f"🤖 Вывод модели:\n{llm_output}")

        # Разбираем действие
        action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
        if not action_match:
            observation = "Ошибка: поле Action не найдено. Убедитесь, что ответ строго следует формату 'Thought: ... Action: ...'."
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*40)
            assistant.prompt_history.append(observation_str)
            continue

        action_str = action_match.group(1).strip()
        tool_name, kwargs = parse_action(action_str)

        # Обработка действия Finish
        if tool_name == "finish":
            final_answer = kwargs.get("answer", "Задача выполнена")
            if display:
                print(f"🎉 Задача выполнена!")
                print(f"📋 Финальный ответ: {final_answer}")
            return final_answer, assistant.prompt_history

        # Обработка вызова инструмента
        if tool_name in available_tools:
            if display:
                print(f"🛠️  Вызов инструмента: {tool_name}({kwargs})")
            observation = available_tools[tool_name](**kwargs)
        else:
            observation = f"Ошибка: неизвестный инструмент '{tool_name}'"

        # Записываем результат наблюдения
        if display:
            print(f"📊 Наблюдение: {observation}")
            print("="*50)

        assistant.add_observation(observation)

    # Если достигнут лимит итераций
    timeout_answer = "Извините, задача не выполнена после нескольких попыток. Попробуйте упростить вопрос или повторите позже."
    if display:
        print(f"⏰ Достигнут лимит итераций: {timeout_answer}")

    return timeout_answer, assistant.prompt_history


In [ ]:
# Тестовый пример
def test_basic_example():
    """Тест: погода в Пекине + рекомендация достопримечательности."""
    print("🚀 Запуск теста: погода в Пекине + достопримечательность")
    user_input = "Привет, проверь сегодняшнюю погоду в Пекине и порекомендуй подходящую достопримечательность."
    
    final_answer, history = run_assistant(user_input, display=True)
    
    print("\n" + "="*60)
    print("📊 Тест завершён!")
    print("="*60)
    print(f"Финальный ответ: {final_answer}")
    
    # Показываем полную историю диалога
    display_conversation(history)
    
    return final_answer, history

# Запускаем тест
final_answer, history = test_basic_example()


In [ ]:
def interactive_travel_assistant():
    """
    Интерактивный режим туристического помощника.
    """
    print("🌍 Добро пожаловать в умного туристического помощника!")
    print("💡 Можете спросить про погоду и достопримечательности любого города")
    print("❌ Введите 'quit' или 'выход' для завершения\n")
    
    while True:
        user_input = input("👤 Ваш вопрос: ").strip()
        
        if user_input.lower() in ['quit', 'выход', 'exit']:
            print("👋 Спасибо за использование помощника, до свидания!")
            break
        
        if not user_input:
            print("⚠️  Пожалуйста, введите вопрос")
            continue
        
        print("\n" + "="*50)
        print("🔄 Обрабатываем ваш запрос...")
        
        final_answer, history = run_assistant(user_input, display=True)
        
        print("\n🎯 Финальный ответ:")
        print("="*30)
        print(final_answer)
        print("="*30)
        
        # Спрашиваем, показывать ли полную историю
        show_history = input("\n📖 Показать полную историю диалога? (y/n): ").strip().lower()
        if show_history in ['y', 'yes', 'да']:
            display_conversation(history)
        
        print("\n" + "="*60)
        print("🔄 Готов к следующему вопросу...\n")

# Быстрый тест
def quick_test(city="Москва"):
    """Быстрый тест для указанного города."""
    user_input = f"Проверь погоду в {city} и порекомендуй подходящие достопримечательности"
    print(f"🚀 Быстрый тест: {user_input}")
    final_answer, _ = run_assistant(user_input, display=True)
    return final_answer


In [ ]:
# Точка входа
if __name__ == "__main__":
    print("Выберите режим:")
    print("1. Тест (Пекин)")
    print("2. Интерактивный режим")
    print("3. Быстрый тест другого города")
    
    choice = input("Введите выбор (1/2/3): ").strip()
    
    if choice == "1":
        test_basic_example()
    elif choice == "2":
        interactive_travel_assistant()
    elif choice == "3":
        city = input("Введите название города: ").strip() or "Москва"
        quick_test(city)
    else:
        print("Неверный выбор, запускаем тест...")
        test_basic_example()
